# Evaluación del modelo clasificador de riesgo IA

En este notebook evaluamos el modelo entrenado sobre el conjunto de test.

Métricas:
1. Classification report (precision, recall, f1 por clase)
2. F1-score macro
3. Matriz de confusión
4. Curva ROC multiclase (One-vs-Rest)
5. Análisis de errores
6. Registro de métricas en MLflow

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Carga del modelo y datos de test

In [ ]:
import pandas as pd
from functions import cargar_artefactos

modelo, tfidf = cargar_artefactos("model")

test_df = pd.read_csv("data/processed/test.csv")
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Test: {len(X_test)} muestras")
print(f"Clases: {sorted(y_test.unique())}")

## 2. Classification report y F1-score macro

In [ ]:
from functions import evaluar_modelo

y_pred, report_dict = evaluar_modelo(modelo, tfidf, X_test, y_test)

## 3. Matriz de confusión

In [ ]:
from functions import mostrar_matriz_confusion

clases = sorted(y_test.unique())
fig_cm = mostrar_matriz_confusion(y_test, y_pred, labels=clases)

## 4. Curva ROC multiclase

In [ ]:
from functions import plot_curva_roc_multiclase

fig_roc, roc_auc_dict = plot_curva_roc_multiclase(modelo, tfidf, X_test, y_test)

## 5. Análisis de errores

In [ ]:
from functions import analisis_errores

df_errores = analisis_errores(modelo, tfidf, X_test, y_test)

## 6. Registro de métricas en MLflow

In [ ]:
import mlflow
import numpy as np

remote_server_uri = "http://34.240.189.163:5000"
mlflow.set_tracking_uri(remote_server_uri)

mlflow.set_experiment("clasificador_riesgo_ia")

with mlflow.start_run(run_name="evaluacion_test"):
    # Métricas globales
    mlflow.log_metric("test_f1_macro", report_dict["macro avg"]["f1-score"])
    mlflow.log_metric("test_accuracy", report_dict["accuracy"])
    mlflow.log_metric("test_precision_macro", report_dict["macro avg"]["precision"])
    mlflow.log_metric("test_recall_macro", report_dict["macro avg"]["recall"])

    # ROC AUC por clase
    for clase, auc_val in roc_auc_dict.items():
        mlflow.log_metric(f"test_roc_auc_{clase}", auc_val)
    mlflow.log_metric("test_roc_auc_macro", np.mean(list(roc_auc_dict.values())))

    # Guardar la matriz de confusión como imagen
    fig_cm.savefig("model/matriz_confusion.png", dpi=150, bbox_inches="tight")
    mlflow.log_artifact("model/matriz_confusion.png")

    # Guardar la curva ROC como imagen
    fig_roc.savefig("model/curva_roc.png", dpi=150, bbox_inches="tight")
    mlflow.log_artifact("model/curva_roc.png")

    print(f"Métricas de test registradas en MLflow:")
    print(f"  F1-score macro: {report_dict['macro avg']['f1-score']:.4f}")
    print(f"  Accuracy: {report_dict['accuracy']:.4f}")
    print(f"  ROC AUC macro: {np.mean(list(roc_auc_dict.values())):.4f}")
    print(f"  Run ID: {mlflow.active_run().info.run_id}")

## 7. Conclusiones

Documentar aquí las conclusiones tras observar las métricas:
- ¿Qué clases se confunden más entre sí?
- ¿El modelo generaliza bien o muestra signos de overfitting?
- ¿Qué mejoras se podrían probar en iteraciones futuras?